# Iter-SPIN — Step 3 (batch, one n per run): generate SPIN answers

Generates the **spin** answers for the coupled sets at the chosen `ONLY_N`. The **baseline** column is reused from a `answers_baseline_*` / `answers_*` CSV if one is attached,
and otherwise **generated here** with the unsuppressed model (self-contained — no dependency on another run). One run does iter/global/percat
at a single n (~3 sets); set `ONLY_N=[2]`, run, commit, then `[3]`, `[5]`, `[10]`. iter n=1 = SPIN is
already done (author-sample spin column). Checkpointed per set. Outputs feed the batch Step 4 judge.

## Config

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, json, time
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'
USE_FAST=False
ONLY_N=[3]                 # <-- which iteration budget to run THIS session. e.g. [2] then [3] then [5] then [10]
ONLY_VARIANTS=['iter','global','percat']
N_EVAL=None                # None = full SALAD (no subsampling)
GEN_BS=4; MAXNEW=64; USE_CHAT_TEMPLATE=False
DTYPE=torch.float16
MODEL_TAG=MODEL_ID.split('/')[-1]
print('model', MODEL_ID, '| ONLY_N', ONLY_N, '| variants', ONLY_VARIANTS)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.5 MB/s eta 0:00:00
model Qwen/Qwen2-7B-Instruct | ONLY_N [3] | variants ['iter', 'global', 'percat']


## SALAD questions + baseline answers (reuse if attached, else generate)

In [2]:
if not os.path.exists('DEAN'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git'], check=True)
DATA='DEAN/src/data'
fair_q=pd.read_csv(f'{DATA}/salad_fairness.csv')['questions'].tolist()[:N_EVAL]
priv_q=pd.read_csv(f'{DATA}/salad_privacy.csv')['questions'].tolist()[:N_EVAL]

def _find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
def baseline_map(cat):
    # reuse a 'baseline' column from any attached answers_*_{cat}.csv if present; else {} -> generated below
    f=_find(f'answers_*_{cat}.csv')
    if not f:
        print(f'  no answers_*_{cat}.csv found -> will GENERATE baseline with the unsuppressed model'); return {}
    df=pd.read_csv(f)
    if 'baseline' not in df.columns:
        print(f'  {os.path.basename(f)} has no baseline column -> will GENERATE'); return {}
    print(f'  baseline for {cat} <- {os.path.basename(f)} ({len(df)} rows)')
    return dict(zip(df['q'].astype(str), df['baseline']))
BASE={'fairness':baseline_map('fairness'),'privacy':baseline_map('privacy')}
missing={c:sum(1 for q in (fair_q if c=='fairness' else priv_q) if str(q) not in BASE[c]) for c in BASE}
print('baseline coverage gaps (0 = reused fully; >0 = will generate below):', missing)

Cloning into 'DEAN'...


  baseline for fairness <- answers_Qwen2-7B-Instruct_authorsample_fairness.csv (2165 rows)
  baseline for privacy <- answers_Qwen2-7B-Instruct_authorsample_privacy.csv (657 rows)
baseline coverage gaps (should be 0): {'fairness': 0, 'privacy': 0}


## Model + generation

In [3]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV)

def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
def suppress_reversible(coupled):
    d=dict(model.named_parameters()); saved=[]; nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); flat=p.data.view(-1)
            saved.append((n,idx,flat[idx].clone())); flat[idx]=0.0; nz+=len(idxlist)
    return saved,nz
def restore(saved):
    d=dict(model.named_parameters())
    for n,idx,vals in saved: d[n].data.view(-1)[idx]=vals
@torch.no_grad()
def gen_spin(queries, ckkey, state, CK):
    done=state.get(ckkey,[]); t0=time.time()
    for i in range(len(done),len(queries),GEN_BS):
        enc=tokenizer(fmt(queries[i:i+GEN_BS]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        g=model.generate(**enc,max_new_tokens=MAXNEW,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        done+=[tokenizer.decode(g[j,enc.input_ids.shape[1]:],skip_special_tokens=True).strip() for j in range(g.shape[0])]
        state[ckkey]=done; json.dump(state,open(CK,'w'))
        print(f'    {ckkey} {len(done)}/{len(queries)} ({time.time()-t0:.0f}s)',end='\r')
    print(); return done
print('gen ready.')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

model loaded | entry cuda:0
gen ready.


## Discover coupled sets for ONLY_N, generate spin (checkpointed), write answers

In [4]:
pats={'iter':'coupled_iter_%s_n*.json'%MODEL_TAG,
      'global':'coupled_control_global_%s_n*.json'%MODEL_TAG,
      'percat':'coupled_control_percat_%s_n*.json'%MODEL_TAG}
SETS=[]
for variant,pat in pats.items():
    if variant not in ONLY_VARIANTS: continue
    seen=set()
    for r in ['.','/kaggle/input','/kaggle/working']:
        for f in glob.glob(os.path.join(r,'**',pat),recursive=True):
            n=int(re.search(r'_n(\d+)\.json$',f).group(1))
            if n in ONLY_N and (variant,n) not in seen:
                seen.add((variant,n)); SETS.append((variant,n,f))
SETS.sort(key=lambda x:(x[1],x[0]))
assert SETS, 'no coupled sets for ONLY_N=%s -- attach the coupled JSONs'%ONLY_N
print('will generate spin for:', [(v,n) for v,n,_ in SETS])

CK='/kaggle/working/gen_spin_ckpt_%s.json'%MODEL_TAG
state=json.load(open(CK)) if os.path.exists(CK) else {}
# --- generate baseline (unsuppressed model) if not fully covered by a reused CSV; runs BEFORE any suppression ---
for cat,qs in [('fairness',fair_q),('privacy',priv_q)]:
    if sum(1 for q in qs if str(q) not in BASE[cat])==0:
        continue
    print(f'[baseline {cat}] generating with unsuppressed model ({len(qs)} q)')
    b=gen_spin(qs, f'baseline_{cat}', state, CK)
    BASE[cat]=dict(zip([str(q) for q in qs], b))
    pd.DataFrame({'q':qs,'baseline':b}).to_csv(f'/kaggle/working/answers_baseline_{MODEL_TAG}_{cat}.csv',index=False)
    print(f'    wrote answers_baseline_{MODEL_TAG}_{cat}.csv')
for variant,n,f in SETS:
    coupled=json.load(open(f))['coupled']
    saved,nz=suppress_reversible(coupled)
    print(f'[{variant} n={n}] zeroed {nz}')
    for cat,qs in [('fairness',fair_q),('privacy',priv_q)]:
        spin=gen_spin(qs, f'{variant}_n{n}_{cat}', state, CK)
        base=[BASE[cat].get(str(q),'') for q in qs]
        out=f'/kaggle/working/answers_{MODEL_TAG}_{variant}_n{n}_{cat}.csv'
        pd.DataFrame({'q':qs,'baseline':base,'spin':spin}).to_csv(out,index=False)
        print(f'    wrote {os.path.basename(out)}')
    restore(saved)
print('\nDONE for ONLY_N', ONLY_N, '-> next session: change ONLY_N and rerun. Then run batch Step 4.')

will generate spin for: [('global', 3), ('iter', 3), ('percat', 3)]
[global n=3] zeroed 3364


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


    global_n3_fairness 2165/2165 (2438s)
    wrote answers_Qwen2-7B-Instruct_global_n3_fairness.csv
    global_n3_privacy 657/657 (742s)
    wrote answers_Qwen2-7B-Instruct_global_n3_privacy.csv
[iter n=3] zeroed 3342
    iter_n3_fairness 2165/2165 (2440s)
    wrote answers_Qwen2-7B-Instruct_iter_n3_fairness.csv
    iter_n3_privacy 657/657 (743s)
    wrote answers_Qwen2-7B-Instruct_iter_n3_privacy.csv
[percat n=3] zeroed 3342
    percat_n3_fairness 2165/2165 (2444s)
    wrote answers_Qwen2-7B-Instruct_percat_n3_fairness.csv
    percat_n3_privacy 657/657 (744s)
    wrote answers_Qwen2-7B-Instruct_percat_n3_privacy.csv

DONE for ONLY_N [3] -> next session: change ONLY_N and rerun. Then run batch Step 4.
